# Session 4 — Multi-Query Attention

All query heads share a single K/V head: maximum cache compression.

In [1]:
import sys; sys.path.insert(0, '../src')
import torch
from kv_cache_variants.attention.mqa import MultiQueryAttention
from kv_cache_variants.attention.mha import MultiHeadAttention
from kv_cache_variants.memory_calc import kv_cache_bytes, human_bytes

torch.manual_seed(0)
d_model, num_heads = 32, 4
x = torch.randn(1, 5, d_model)
_, kv_mha = MultiHeadAttention(d_model, num_heads)(x)
_, kv_mqa = MultiQueryAttention(d_model, num_heads)(x)
print('MHA cached heads:', kv_mha[0].shape[1], '| MQA cached heads:', kv_mqa[0].shape[1])

MHA cached heads: 4 | MQA cached heads: 1


In [2]:
head_dim = d_model // num_heads
mha_bytes = kv_cache_bytes(num_layers=32, num_kv_heads=num_heads, head_dim=head_dim, seq_len=4096)
mqa_bytes = kv_cache_bytes(num_layers=32, num_kv_heads=1, head_dim=head_dim, seq_len=4096)
print(f'MHA: {human_bytes(mha_bytes)} vs MQA: {human_bytes(mqa_bytes)}')

MHA: 16.00 MB vs MQA: 4.00 MB
